# 📘 Aula 07a: Visão Computacional Clássica – MNIST com MLP Flat no Keras & Pipelines Scikit-Learn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula07a_mnist_flat_keras.ipynb)

Seja bem-vindo ao primeiro caderno do nosso módulo de **Visão Computacional e Reconhecimento de Imagens**!

Nesta aula, abordaremos o clássico e histórico dataset **MNIST (Modified National Institute of Standards and Technology)**, considerado o *"Hello World"* do Deep Learning e da Visão Computacional. Utilizaremos uma abordagem baseada em **Redes Neurais Totalmente Conectadas (Dense / MLP Flat)**, onde as imagens bidimensionais de $28 \times 28$ pixels são "achatadas" em vetores unidimensionais de 784 valores contínuos.

---

### 🎯 Objetivo da Aula
1. Carregar e inspecionar a estrutura matricial do dataset MNIST (60.000 imagens de treino e 10.000 de teste).
2. Visualizar os dígitos manuscritos e sua distribuição no plano com o Matplotlib.
3. Compreender a transformação geométrica de *Achatamento (Flattening)* de tensores 2D ($28 \times 28$) para vetores 1D ($784$).
4. Explorar as propriedades matemáticas da ativação não-linear **ReLU (Rectified Linear Unit)**.
5. Treinar um classificador neural denso (`Dense(512) -> Dense(10)`) com Keras.
6. **A Pegadinha do Pré-processamento**: Entender por que o `StandardScaler` clássico falha em imagens e construir transformadores customizados do Scikit-Learn (`Divide255` e `Flatten`).
7. Construir **Pipelines integrados** do Scikit-Learn capazes de alimentar tanto redes Keras quanto modelos de ensemble clássicos (`RandomForestClassifier`).

---

### 💡 A Intenção Pedagógica do Professor
* **Achatamento (Flattening)**: O professor inicia a visão computacional pelo modelo *Flat* para demonstrar a limitação de tratar imagens como simples vetores de números, perdendo a correlação espacial 2D (motivação que nos levará às Convoluções na Aula 07b!).
* **A Pegadinha do StandardScaler**: Demonstrar por que aplicar padronização Z-score em pixels gera anomalias (muitos pixels de borda têm variância zero) e por que a divisão direta por 255.0 é o padrão ouro na indústria de visão computacional.
* **Design Limpo com Pipeline**: Demonstrar como criar classes derivadas de `TransformerMixin` para encapsular pré-processamento dentro de um `Pipeline`, eliminando qualquer risco de vazamento de dados (*data leakage*).

---

### 📚 Glossário da Aula
* **MNIST**: Coleção de 70.000 dígitos manuscritos de 0 a 9 em escala de cinza ($28 \times 28$ pixels, valores de 0 a 255).
* **Flatten (Achatamento)**: Operação que transforma uma matriz $28 \times 28$ em um vetor linear de dimensão $784$.
* **ReLU**: $\max(0, x)$ — função de ativação que zera valores negativos e mantém valores positivos de forma linear, permitindo gradientes não saturantes.
* **Sparse Categorical Crossentropy**: Função de perda para classificação multiclasse que aceita rótulos diretamente como inteiros ($0, 1, ..., 9$), dispensando conversão prévia para One-Hot.
* **TransformerMixin**: Classe base do Scikit-Learn para implementar transformadores customizados compatíveis com o método `.fit_transform()`.

---

### 🧭 Trilha da Aula
1. [Carregamento do Dataset MNIST](#1-carregamento-do-dataset-mnist)
2. [Visualização de Amostras com Matplotlib](#2-visualização-de-um-dígito-individual)
3. [Grid 3x3 de Dígitos Aleatórios](#3-amostragem-aleatória-em-grade-3x3)
4. [Achatamento (Flatten) para Vetores de 784 Pixels](#4-achatamento-dimensional-das-imagens)
5. [A Função de Ativação ReLU](#5-intuição-e-gráfico-da-ativação-relu)
6. [Primeiro Modelo Keras MLP (Dense 512 -> Dense 10)](#6-modelo-sequencial-com-sparse-categorical-crossentropy)
7. [Inspeção de Predições com Argmax](#7-inspeção-de-predições-e-probabilidades)
8. [Encapsulamento no Scikit-Learn: KerasWrappedNN](#8-encapsulador-profissional-keraswrappednn)
9. [A Armadilha do StandardScaler em Imagens](#9-a-armadilha-do-standardscaler-em-imagens)
10. [Transformadores Customizados: Divide255 e Flatten](#10-transformadores-customizados-divide255-e-flatten)
11. [Pipeline Completo com Keras e Random Forest](#11-pipelines-completos-keras-vs-random-forest)


## 1. Carregamento do Dataset MNIST

**O que estamos fazendo**: Carregar os dados oficiais do MNIST disponibilizados pelo Keras, contendo 60.000 imagens de treino e 10.000 de teste.

**Por que fazer assim**: O MNIST é a referência clássica para benchmarking de visão computacional.

> [!NOTE]
> O formato `train_images.shape` é `(60000, 28, 28)`: um tensor 3D de 60 mil matrizes $28 \times 28$ com valores inteiros de $0$ (fundo preto) a $255$ (traço branco).



In [ ]:
# Importa o módulo oficial de carregamento do MNIST pelo Keras
from tensorflow.keras.datasets import mnist

# Baixa e carrega a partição de 60.000 imagens de treino e 10.000 de teste
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Inspeciona o formato dimensional dos tensores
print('Dimensões do tensor de imagens de treino:', train_images.shape)
print('Quantidade de rótulos de treino:', len(train_labels))
print('Classes únicas presentes nos rótulos:', set(train_labels))


## 2. Visualização de um Dígito Individual

**O que estamos fazendo**: Plotar a primeira imagem do dataset utilizando Matplotlib com mapa de cores binário (`plt.cm.binary`).

**Por que fazer assim**: Permite inspecionar a qualidade e o traço do manuscrito original.



In [ ]:
# Importa pyplot do Matplotlib para renderização gráfica
from matplotlib import pyplot as plt

# Seleciona a primeira imagem do conjunto de treino (matriz 28x28)
digit = train_images[0]

# Renderiza a matriz como imagem em escala de cinza/binária
plt.figure(figsize=(4, 4))
plt.imshow(digit, cmap=plt.cm.binary)
plt.title('Amostra de Imagem 0 do MNIST')
plt.axis('off')
plt.show()


## 3. Rótulo da Primeira Imagem

**O que estamos fazendo**: Verificar o rótulo numérico correspondente à imagem que acabamos de visualizar.

**Por que fazer assim**: Confirma se a anotação supervisionada bate com o dígito desenhado (o número 5).



In [ ]:
# Exibe o rótulo numérico supervisionado da primeira amostra
print(f'Rótulo da primeira amostra: {train_labels[0]}')


## 4. Amostragem Aleatória em Grade 3x3

**O que estamos fazendo**: Sortear 9 imagens aleatórias do conjunto de treino e desenhá-las em um grid $3 \times 3$ com seus respectivos títulos.

**Por que fazer assim**: A visualização diversificada revela a variabilidade natural de caligrafias humanas presentes no dataset.

> [!TIP]
> `np.random.randint(0, train_images.shape[0], 9)` sorteia 9 índices não tendenciosos.



In [ ]:
# NumPy para sorteio aleatório
import numpy as np

# Sorteia 9 índices aleatórios do conjunto de 60.000 imagens
samples = np.random.randint(0, train_images.shape[0], 9)

plt.figure(figsize=(6, 6))
for (i, sample) in enumerate(samples):
  # Cria um subplot na posição i + 1 do grid 3x3
  plt.subplot(3, 3, i + 1)
  digit = train_images[sample]
  plt.imshow(digit, cmap=plt.cm.binary)
  plt.title(f'Rótulo: {train_labels[sample]}')
  plt.axis('off')
plt.tight_layout()
plt.show()


## 5. Achatamento Dimensional das Imagens (Flattening)

**O que estamos fazendo**: Redimensionar as matrizes $28 \times 28$ para vetores contínuos de $784$ elementos ($28 \times 28 = 784$).

**Por que fazer assim**: Camadas densas padrão (`Dense`) recebem vetores 1D como entrada, exigindo o achatamento prévio da matriz 2D.

> [!NOTE]
> O parâmetro `-1` no reshape faz o NumPy calcular automaticamente a dimensão do lote ($60.000$ e $10.000$).

> [!WARNING]
> Ao achatar a imagem, perdemos a proximidade espacial 2D (um pixel na linha de cima agora fica a 28 posições de distância do pixel logo abaixo dele!).



In [ ]:
# Achata a matriz 28x28 para um vetor unidimensional de 784 posições
train_flat_images = train_images.reshape((-1, 28 * 28))
test_flat_images = test_images.reshape((-1, 28 * 28))

print('Dimensão de train_flat_images:', train_flat_images.shape)
print('Dimensão de test_flat_images:', test_flat_images.shape)


## 6. Contagem de Classes

**O que estamos fazendo**: Verificar a cardinalidade do conjunto de classes.

**Por que fazer assim**: Para dígitos de 0 a 9, temos exatamente 10 classes.



In [ ]:
# Quantidade de classes distintas no problema
print(f'n_classes: {len(set(train_labels))}')


## 7. Intuição e Gráfico da Ativação ReLU

**O que estamos fazendo**: Plotar a função $\text{ReLU}(x) = \max(0, x)$ no intervalo $[-5, 5]$.

**Por que fazer assim**: A ReLU é a ativação padrão do Deep Learning moderno por eliminar a saturação e acelerar dramaticamente a convergência do gradiente.

> [!NOTE]
> Para $x \le 0$, a derivada é $0$. Para $x > 0$, a derivada é $1.0$, o que evita o problema do desvanecimento de gradientes (*vanishing gradients*)!



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Definição da função ReLU: máximo entre zero e o valor de entrada
def relu(x):
  return np.maximum(0, x)

# Gera 100 pontos entre -5 e +5
x_values = np.linspace(-5, 5, 100)
y_values = relu(x_values)

plt.figure(figsize=(6, 4))
plt.plot(x_values, y_values, 'b-', linewidth=2, label='ReLU(x) = max(0, x)')
plt.title('Função de Ativação ReLU (Rectified Linear Unit)')
plt.xlabel('x (Entrada Linear)')
plt.ylabel('Ativação de Saída')
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
plt.legend()
plt.show()


## 8. Modelo Sequencial com Sparse Categorical Crossentropy

**O que estamos fazendo**: Construir uma rede com 512 neurônios ocultos (ReLU) e 10 de saída (Softmax), treinando com a perda `sparse_categorical_crossentropy`.

**Por que fazer assim**: Diferente da `categorical_crossentropy`, a versão `sparse` aceita os rótulos de classe diretamente como inteiros ($0, 1, ..., 9$) sem exigir conversão One-Hot!

> [!TIP]
> Treinamos por 5 épocas com minibatches de 128 amostras utilizando o otimizador RMSprop.



In [ ]:
# Importa Keras e a camada Dense
from tensorflow import keras
from tensorflow.keras import layers

# Cria a arquitetura sequencial totalmente conectada (MLP)
model = keras.Sequential([
  # Camada Oculta: 512 neurônios com ativação ReLU
  layers.Dense(512, activation='relu'),
  # Camada de Saída: 10 neurônios com Softmax para probabilidades
  layers.Dense(10, activation='softmax')
])

# Compila com sparse_categorical_crossentropy (rótulos inteiros diretos)
model.compile(
  optimizer='rmsprop',
  loss='sparse_categorical_crossentropy',
  metrics=['accuracy']
)

# Treina o modelo nas imagens achatadas
model.fit(
  train_flat_images, train_labels,
  epochs=5,
  batch_size=128
)

# Avalia nos dados de teste não vistos
test_loss, test_acc = model.evaluate(test_flat_images, test_labels)
print(f'Acurácia no Teste: {test_acc * 100:.2f}%')


## 9. Inspeção das Probabilidades Preditas

**O que estamos fazendo**: Gerar a distribuição de probabilidades para as 5 primeiras amostras de treino.

**Por que fazer assim**: Cada linha contém as 10 probabilidades calculadas pela Softmax.



In [ ]:
# Realiza a inferência para as 5 primeiras amostras
y_pred = model.predict(train_flat_images[:5])
print('Probabilidades estimadas para as 5 primeiras imagens (formato 5x10):
', y_pred)


## 10. Decodificação via Argmax e Comparação

**O que estamos fazendo**: Obter o índice de maior probabilidade para cada amostra com `np.argmax` e comparar com os rótulos reais.

**Por que fazer assim**: Confirma na prática que o modelo acerta com altíssima confiança os dígitos apresentados.



In [ ]:
# Extrai o dígito de maior probabilidade predito
print('Dígitos preditos (argmax):', np.argmax(y_pred, axis=1))
# Exibe os rótulos reais correspondentes
print('Rótulos reais esperados:  ', train_labels[:5])


## 11. Encapsulador Profissional Scikit-Learn: KerasWrappedNN

**O que estamos fazendo**: Criar uma classe compatível com a API do Scikit-Learn (`BaseEstimator`, `ClassifierMixin`) que aceita uma fábrica de modelos (`model_fabric`).

**Por que fazer assim**: Permite conectar qualquer modelo Keras aos pipelines, seletores de hiperparâmetros e avaliadores cruzados do ecossistema Scikit-Learn.

> [!NOTE]
> A fábrica `BasicKerasNN` permite trocar a arquitetura da rede sem alterar a lógica de treinamento e predição.



In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
import numpy as np

# Função de fábrica que instancia a arquitetura da rede Keras
def BasicKerasNN(num_classes):
  return keras.Sequential([
      layers.Dense(512, activation='relu'),
      layers.Dense(num_classes, activation='softmax')
    ])

class KerasWrappedNN(BaseEstimator, ClassifierMixin):
  """
  Wrapper para modelos Keras totalmente compatível com a API do Scikit-Learn.
  """
  def __init__(self, epochs, batch_size, model_fabric=BasicKerasNN):
    self.epochs = epochs
    self.batch_size = batch_size
    self.model_fabric = model_fabric

  def fit(self, X, y):
    # Identifica classes únicas e gera representação One-Hot
    self.labels, ids = np.unique(y, return_inverse=True)
    y_hot = keras.utils.to_categorical(ids, len(self.labels))
    # Cria o modelo via função de fábrica
    self.model = self.model_fabric(len(self.labels))
    self.model.compile(
      optimizer='rmsprop',
      loss='categorical_crossentropy',
      metrics=['accuracy']
    )
    # Executa o ajuste dos pesos
    self.model.fit(
      X, y_hot,
      epochs=self.epochs,
      batch_size=self.batch_size
    )
    return self

  def predict(self, X):
    # Gera predições de probabilidade e retorna a classe de maior probabilidade
    y_pred = self.model.predict(X)
    return self.labels[np.argmax(y_pred, axis=1)]

# Instancia e treina o wrapper Keras
model = KerasWrappedNN(epochs=5, batch_size=128)
model.fit(train_flat_images, train_labels)
y_pred = model.predict(test_flat_images)
print(f'Acurácia KerasWrappedNN: {accuracy_score(test_labels, y_pred) * 100:.2f}%')


## 12. A Armadilha do StandardScaler em Imagens

**O que estamos fazendo**: Inserir um `StandardScaler()` clássico no Pipeline antes da rede neural.

**Por que fazer assim**: Demonstração pedagógica fundamental: vamos ver o que acontece quando tentamos usar padronização estatística tradicional em imagens.

> [!WARNING]
> Muitos pixels das bordas de imagens são constantes (sempre 0). Em pixels com variância zero, o StandardScaler gera divisão por zero ou artefatos numéricos indesejados!



In [ ]:
# Importa Pipeline e StandardScaler do Scikit-Learn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Monta o pipeline com padronização Z-score tradicional
model = Pipeline([
  ('scaler', StandardScaler()),
  ('nn', KerasWrappedNN(epochs=5, batch_size=128))
])
model.fit(train_flat_images, train_labels)
y_pred = model.predict(test_flat_images)
print(f'Acurácia com StandardScaler: {accuracy_score(test_labels, y_pred) * 100:.2f}%')


## 13. Inspeção dos Pixels Brutos

**O que estamos fazendo**: Visualizar o vetor numérico do primeiro dígito para observar a grande quantidade de valores zero nas bordas.

**Por que fazer assim**: Explica o comportamento anômalo do StandardScaler.



In [ ]:
# Visualiza os valores brutos do primeiro vetor de 784 pixels
print('Vetor bruto do dígito 0 (784 valores):
', train_flat_images[0])


## 14. O Efeito do StandardScaler nos Pixels

**O que estamos fazendo**: Ajustar o StandardScaler e transformar a primeira amostra.

**Por que fazer assim**: Observe como os pixels pretos das bordas (que eram 0) viram valores distorcidos em vez de preservarem o fundo limpo da imagem.



In [ ]:
scaler = StandardScaler()
scaler.fit(train_flat_images)
train_flat_image0_scaled = scaler.transform(train_flat_images[:1])
print('Primeira amostra após StandardScaler (observe as distorções nos zeros):
', train_flat_image0_scaled)


## 15. A Solução Correta: Escalar por 255.0

**O que estamos fazendo**: Dividir a intensidade dos pixels por 255.0.

**Por que fazer assim**: Como imagens de 8 bits têm intensidade entre 0 e 255, dividir por 255 projeta os valores perfeitamente no intervalo $[0.0, 1.0]$, preservando a esparsidade dos fundos pretos (0 continua sendo 0.0)!

> [!TIP]
> Dividir por 255.0 é uma operação linear sem deslocamento de média, mantendo intacta a estrutura natural das imagens.



In [ ]:
# Projeção linear direta no intervalo [0, 1]
scaled_sample = train_flat_images[:1] / 255.0
print('Amostra normalizada dividindo por 255.0:
', scaled_sample)


## 16. Criando o Transformador Customizado Divide255

**O que estamos fazendo**: Criar uma classe `Divide255` derivada de `TransformerMixin` para encapsular a divisão por 255 dentro de um `Pipeline`.

**Por que fazer assim**: O Scikit-Learn exige a interface com métodos `.fit()` e `.transform()`. Criando essa classe, a normalização correta vira uma etapa nativa do Pipeline.



In [ ]:
# Classe base para transformadores do Scikit-Learn
from sklearn.base import TransformerMixin

class Divide255(TransformerMixin):
  """
  Transformador Scikit-Learn que normaliza imagens dividindo todos os pixels por 255.0.
  """
  def fit(self, X, y=None):
    return self

  def transform(self, X, y=None):
    return X / 255.0

# Testa o Pipeline com Divide255 no lugar do StandardScaler
model = Pipeline([
  ('scaler', Divide255()),
  ('nn', KerasWrappedNN(epochs=5, batch_size=128))
])
model.fit(train_flat_images, train_labels)
y_pred = model.predict(test_flat_images)
print(f'Acurácia com Pipeline (Divide255): {accuracy_score(test_labels, y_pred) * 100:.2f}%')


## 17. Criando o Transformador Customizado Flatten

**O que estamos fazendo**: Criar a classe `Flatten(TransformerMixin)` para embutir o achatamento $28 \times 28 \rightarrow 784$ dentro do Pipeline.

**Por que fazer assim**: Agora podemos passar diretamente as imagens originais em formato 2D `train_images` para o Pipeline, e ele cuidará de normalizar E achatar automaticamente!

> [!TIP]
> O Pipeline cuida de todo o fluxo de ponta a ponta sem precisarmos criar variáveis intermediárias manuais!



In [ ]:
class Flatten(TransformerMixin):
  """
  Transformador Scikit-Learn que achata tensores multidimensionais para vetores 1D por amostra.
  """
  def fit(self, X, y=None):
    return self

  def transform(self, X, y=None):
    # Transforma (N, 28, 28) em (N, 784)
    return X.reshape((X.shape[0], -1))

# Pipeline completo de ponta a ponta: Divide255 -> Flatten -> Rede Neural Keras
model = Pipeline([
  ('scaler', Divide255()),
  ('flatten', Flatten()),
  ('nn', KerasWrappedNN(epochs=5, batch_size=128))
])
# Observe que passamos train_images e test_images em sua forma 2D original!
model.fit(train_images, train_labels)
y_pred = model.predict(test_images)
print(f'Acurácia do Pipeline Completo (Keras): {accuracy_score(test_labels, y_pred) * 100:.2f}%')


## 18. Comparativo com Random Forest no Mesmo Pipeline

**O que estamos fazendo**: Substituir o estimador neural por um `RandomForestClassifier()` mantendo os mesmos transformadores `Divide255` e `Flatten`.

**Por que fazer assim**: Comprova o poder do desacoplamento arquitetural: o mesmo pipeline de pré-processamento alimenta tanto redes neurais quanto algoritmos clássicos de árvores de decisão ensacadas (*bagging*).

> [!NOTE]
> A floresta aleatória atinge excelente acurácia (~97%), mas sua memória e tempo de inferência escalam com o número de árvores, enquanto a rede neural executa multiplicações de matrizes rápidas em GPU.



In [ ]:
# Importa classificador Random Forest do Scikit-Learn
from sklearn.ensemble import RandomForestClassifier

# Constrói o pipeline reaproveitando Divide255 e Flatten com Random Forest
model = Pipeline([
  ('scaler', Divide255()),
  ('flatten', Flatten()),
  ('rf', RandomForestClassifier())
])

# Treina a floresta aleatória nos dados de imagem originais
model.fit(train_images, train_labels)
# Avalia no teste
y_pred = model.predict(test_images)
print(f'Acurácia do Pipeline com Random Forest: {accuracy_score(test_labels, y_pred) * 100:.2f}%')


---

## 🎓 Resumo da Aula & O Que Você Aprendeu

Nesta aula, realizamos uma imersão completa na Visão Computacional clássica utilizando modelos totalmente conectados (MLP Flat) integrados ao Scikit-Learn:

1. **Exploração do MNIST**: Carregamos e visualizamos os $70.000$ dígitos manuscritos $28 \times 28$ em escala de cinza.
2. **Achatamento (Flatten)**: Vimos como transformar matrizes $28 \times 28$ em vetores de $784$ elementos para alimentar camadas densas.
3. **Ativação ReLU**: Compreendemos por que $\max(0, x)$ é a função de ativação preferida em redes profundas (gradiente constante e sem saturação).
4. **A Armadilha do StandardScaler**: Descobrimos por que a padronização Z-score falha em imagens com fundos pretos esparsos e construímos o transformador `Divide255` para projetar os pixels no intervalo $[0.0, 1.0]$.
5. **Pipelines Profissionais no Scikit-Learn**: Criamos `Divide255` e `Flatten` como `TransformerMixin`, conectando imagens cruas a modelos Keras (`KerasWrappedNN`) e florestas aleatórias (`RandomForestClassifier`) com total compatibilidade.

---

## 👉 Próximo Passo
Na **Aula 07b**, daremos o grande salto da visão computacional moderna: abandonaremos o achatamento para preservar a geometria 2D das imagens usando **Redes Neurais Convolucionais (CNNs)** com `Conv2D` e `MaxPooling2D` no Keras!
